
# ROSMAP DLPFC Phenotype data 

- `input`:    
    1. today data from xqtl-protocal : for alignment pipeline testing
    2. 1141 ROSMAP DLPFC junc files from BU (the upstream bam files have not been precessed with WASP yet)


- `output`:
    1. splicing events from leafcutter2
    2. the phenotype data for association anlaysis


## 0. data preparation

For this part, I am using the toydata for the testing of new STAR alignment process.... 

### RNA Seq Alignment
see [previous work ](https://cumc.github.io/xqtl-pipeline/code/molecular_phenotypes/bulk_expression.html)

### Perform data quality summary via `fastqc`
see [previous work ](https://cumc.github.io/xqtl-pipeline/code/molecular_phenotypes/bulk_expression.html)

### Cut adaptor (Optional)
see [previous work ](https://cumc.github.io/xqtl-pipeline/code/molecular_phenotypes/bulk_expression.html)
This step will trim the fastq file to remove the adaptor. It is optional because the fastq in the protocol data folders are converted from bam file and are already without adaptors.


### Read alignment via STAR with WASP and QC via Picard (toydata for pipeline test)

## 1. Phenotype data: Leaf_cutter2

**For this part, I am using the junc files from BU with STAR alignment without WASP**

In [6]:
cd /home/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024
ln -s ~/codes/xqtl-pipeline/pipeline/ ./


[1]+  Done                    nohup python /home/rf2872/codes/leafcutter2/scripts/leafcutter2_regtools.py -A ~/data/ref_data_Ru/gencode.v45.basic.annotation.gtf.gz -j output/leafcutter2/ROSMAP_DLPFC_intron_usage_perind.junc -G /mnt/vast/hpc/csg/rf2872/data/ref_data_Ru/hg38.fa.gz -o ROSMAP_DLPFC_const -r output/leafcutter2_const --includeconst &> leafcutter2_const.log
ln: failed to create symbolic link './pipeline': File exists


: 1

### prepare whole junc file for analysis



In [24]:
sos run  pipeline/splicing_normalization.ipynb Junc_list \
    --junc_path ~/Work/leaf_cutter2/ROSMAP_DLPFC_2024/junc_files


INFO: Running Junc_list: 
INFO: Junc_list is completed.
INFO: Junc_list output:   /mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/leafcutter2/ROSMAP_DLPFC_intron_usage_perind.junc
INFO: Workflow Junc_list (ID=w474f8ddd0f03cf56) is executed successfully with 1 completed step.


### (Optional) remove duplicates in sample list

In [9]:
library(tidyverse)
sam_list_path <- '/mnt/vast/hpc/csg/ftp_lisanwanglab_sync/ftp_fgc_xqtl/projects/rna-seq/BU/ROSMAP_DLPFC/rosmap_dlpfc_duplicates.tsv'
lookup_path <- '~/xqtl_data/ROSMAP/DLPFC/eQTL/sample_map.txt'

sam_list <- read_delim(sam_list_path)
sam_list_no <- sam_list %>% filter(keep == "no")

lookup <- read_delim(lookup_path)
lookup %>% dim

lookup_nodup <- lookup %>% filter(!(sample_id %in% (sam_list_no %>% pull(sample_id))))
dim(lookup_nodup)

#save the new phenotype data
write_delim(lookup_nodup, lookup_path %>% gsub(".txt",".remove_duplicates.txt",.), delim = '\t',) 

### prepare sample list for analysis

`sample_id` is the ID for RNAseq/bam file preffix, and `participant_id` is the ID for WGS/geno file preffix. Here I use the sample_lookup file from Hao, but I need to cut it to the same length as my files first.

In [25]:
sos run pipeline/splicing_normalization.ipynb Jointcall_samples \
    --sample_table sample_map.remove_duplicates.txt  \
    --junc_list output/leafcutter2/ROSMAP_DLPFC_intron_usage_perind.junc


INFO: Running Jointcall_samples: 
INFO: Jointcall_samples is completed.
INFO: Jointcall_samples output:   /mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/leafcutter2/sample_map.remove_duplicates.txt.rnaseq
INFO: Workflow Jointcall_samples (ID=wb1a46109d7b5eb33) is executed successfully with 1 completed step.


### run the leafcutter2 script to generate leafcutter2 outputs:
*  `input`: the "*refined_noisy" out file that includes noisy introns from previous step, -N annotation files `gencode_v43_plus_v37_productive.intron_by_transcript_BEDlike.txt.gz` providing 'functional' or 'productive' info from author,  the "*intron_usage_perind.junc " file from previous and previous step 
*  `output`: different type of leafcutter2_perind.counts.* files. while leafcutter2_perind.counts.noise_by_intron.gz has 5 columns splited by ":" in chrom. 

    - {out_prefix}_perind.counts.noise.gz: output functional introns (intact), and 
                  noisy introns. Note the start and end coordinates of noisy introns are recalibrated
                  to the min(starts) and max(ends) of all functional introns within cluster.
    - {out_prefix}_perind_numers.counts.noise.gz: same as above, except write numerators.
    - {out_prefix}_perind.counts.noise_by_intron.gz: same as the first output, except here
                  noisy introns' coordinates are kept as their original coordinates.  

In [29]:
nohup python /home/rf2872/codes/leafcutter2/scripts/leafcutter2_regtools.py \
    -A ~/data/ref_data_Ru/gencode.v45.basic.annotation.gtf.gz \
    -j output/leafcutter2/ROSMAP_DLPFC_intron_usage_perind.junc \
    -G /mnt/vast/hpc/csg/rf2872/data/ref_data_Ru/hg38.fa.gz\
    -o ROSMAP_DLPFC \
    -r output/leafcutter2 &> leafcutter2.log &

[1]+  Exit 2                  nohup python /home/rf2872/codes/leafcutter2/scripts/leafcutter2_regtools.py -j output/leafcutter2/ROSMAP_DLPFC_intron_usage_perind.junc -N /home/rf2872/codes/leafcutter2/data/gencode_v43_plus_v37_productive.intron_by_transcript_BEDlike.txt.gz -o ROSMAP_DLPFC -r output/leafcutter2 -A ~/data/ref_data_Ru/gencode.v45.basic.annotation.gtf.gz &> leafcutter2.log
[1] 8684


### (Optional) constitutive version: the leafcutter2 script to generate leafcutter2 outputs:
*  run a version with --includeconst flag. this output all the constitutive introns. constitutive introns are clusters that only ever have 1 intron. These type of introns (or resulting exon junctions) are overwhelmingly productive protein coding. with out the flag, if a cluster only has a single intron, then that intron is not included in the output. Imagine you have a gene with 2 exons only, and they are not alternatively spliced, then you will always have the same intron. Since there are no alternatvie introns here, the cluster formed alwyas include this single intron.

In [5]:
nohup python /home/rf2872/codes/leafcutter2/scripts/leafcutter2_regtools.py \
    -A ~/data/ref_data_Ru/gencode.v45.basic.annotation.gtf.gz \
    -j output/leafcutter2/ROSMAP_DLPFC_intron_usage_perind.junc \
    -G /mnt/vast/hpc/csg/rf2872/data/ref_data_Ru/hg38.fa.gz\
    -o ROSMAP_DLPFC_const \
    -r output/leafcutter2_const --includeconst&> leafcutter2_const.log &

[1] 6428


### (Optional) Filtering the output of leafcutter2

### QC and Normalization of leafCutter outputs
*  `input`: the "_intron_usage_perind.counts.gz" file from previous step # here I use _perind.counts.noise.gz, which has the same format with leaf_cutter
*  `output`: QC'd and normalized phenotype table end with "qqnorm.txt"
Be noted that the `ratio` file to be fed into the leafcutter_norm are the one without `number` tag in its filename. 

In [ ]:
sos run  ~/codes/xqtl-pipeline/pipeline/splicing_normalization.ipynb leafcutter_norm \
    --cwd output/leafcutter2/ \
    --ratios output/leafcutter2/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz \
    --container oras://ghcr.io/cumc/leafcutter_apptainer:latest 
# in new version should add --pseudo_ratio 

In [ ]:
cd /mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/leafcutter2/count0.5

sos run  ~/codes/xqtl-pipeline/pipeline/splicing_normalization.ipynb leafcutter_norm \
    --cwd . \
    --ratios ./ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz \
    --container oras://ghcr.io/cumc/leafcutter_apptainer:latest 

In [18]:
library(data.table)
library(tidyverse)
df <- fread('/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/leafcutter2/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.txt')

In [7]:
df_orig <- fread('/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/leafcutter2/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.txt.original')
df_0.5count <- fread('/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/leafcutter2/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.txt')


In [20]:
df %>% filter(ID == 'chr18:166819:178932:clu_149502_+:PR')

#Chr,start,end,ID,01_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,02_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,03_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,04_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,05_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,07_120410.Aligned.sortedByCoord.out_wasp_qc.md.junc,⋯,RISK_95.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_97.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_99.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_9_rerun.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T6Z5.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T717.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-49KVA.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYO4G.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOQN.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOSS.Aligned.sortedByCoord.out_wasp_qc.md.junc
<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr18,166819,178932,chr18:166819:178932:clu_149502_+:PR,-2.263613,0.91572,-1.158344,0.8603896,-1.599872,0.2118575,⋯,0.1335666,-1.807857,0.1892235,-0.6911483,0.6698242,0.4160287,0.7251579,0.8480507,0.6578172,0.8310074


In [8]:
df_orig %>% filter(ID == 'chr18:166819:178932:clu_149502_+:PR')

#Chr,start,end,ID,01_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,02_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,03_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,04_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,05_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,07_120410.Aligned.sortedByCoord.out_wasp_qc.md.junc,⋯,RISK_95.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_97.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_99.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_9_rerun.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T6Z5.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T717.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-49KVA.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYO4G.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOQN.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOSS.Aligned.sortedByCoord.out_wasp_qc.md.junc
<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr18,166819,178932,chr18:166819:178932:clu_149502_+:PR,-2.246489,0.9706492,-1.159394,0.8906807,-1.68357,-0.2129666,⋯,-0.08033276,-1.910126,-0.04298344,-0.4568178,0.3267873,-0.1835575,0.287428,0.6324037,0.2757295,0.5956064


In [9]:
df_0.5count %>% filter(ID == 'chr18:166819:178932:clu_149502_+:PR')

#Chr,start,end,ID,01_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,02_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,03_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,04_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,05_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,07_120410.Aligned.sortedByCoord.out_wasp_qc.md.junc,⋯,RISK_95.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_97.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_99.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_9_rerun.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T6Z5.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T717.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-49KVA.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYO4G.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOQN.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOSS.Aligned.sortedByCoord.out_wasp_qc.md.junc
<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr18,166819,178932,chr18:166819:178932:clu_149502_+:PR,-2.246489,0.9706492,-1.159394,0.8906807,-1.68357,-0.2129666,⋯,-0.08033276,-1.910126,-0.04298344,-0.4568178,0.3267873,-0.1835575,0.287428,0.6324037,0.2757295,0.5956064


In [12]:
normlized_data_1per <- fread('/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.gz')
normlized_data_0.5count <- fread('/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/normalize_impute_back/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.gz')

In [13]:
normlized_data_1per %>% filter(ID == 'chr18:166819:178932:clu_149502_+:PR')
normlized_data_0.5count %>% filter(ID == 'chr18:166819:178932:clu_149502_+:PR')

#Chr,start,end,ID,01_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,02_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,03_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,04_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,05_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,07_120410.Aligned.sortedByCoord.out_wasp_qc.md.junc,⋯,RISK_95.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_97.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_99.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_9_rerun.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T6Z5.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T717.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-49KVA.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYO4G.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOQN.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOSS.Aligned.sortedByCoord.out_wasp_qc.md.junc
<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr18,166819,178932,chr18:166819:178932:clu_149502_+:PR,-2.263613,0.91572,-1.158344,0.8603896,-1.599872,0.2118575,⋯,0.1335666,-1.807857,0.1892235,-0.6911483,0.6698242,0.4160287,0.7251579,0.8480507,0.6578172,0.8310074


#Chr,start,end,ID,01_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,02_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,03_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,04_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,05_120405.Aligned.sortedByCoord.out_wasp_qc.md.junc,07_120410.Aligned.sortedByCoord.out_wasp_qc.md.junc,⋯,RISK_95.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_97.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_99.Aligned.sortedByCoord.out_wasp_qc.md.junc,RISK_9_rerun.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T6Z5.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-2T717.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-49KVA.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYO4G.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOQN.Aligned.sortedByCoord.out_wasp_qc.md.junc,SM-AYOSS.Aligned.sortedByCoord.out_wasp_qc.md.junc
<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr18,166819,178932,chr18:166819:178932:clu_149502_+:PR,-2.246489,0.9706492,-1.159394,0.8906807,-1.68357,-0.2129666,⋯,-0.08033276,-1.910126,-0.04298344,-0.4568178,0.3267873,-0.1835575,0.287428,0.6324037,0.2757295,0.5956064


### Imputation


In [ ]:
sos run pipeline/phenotype_imputation.ipynb EBMF \
    --phenoFile output/leafcutter2/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.txt \
    --cwd output/normalize_impute \
    --prior ebnm_point_laplace --varType 1 \
    --container oras://ghcr.io/cumc/factor_analysis_apptainer:latest \
    --mem 40G \
    --numThreads 20 \
    --walltime 100h \
    -c ~/env_files/csg.yml 

### Normalize after imputation (for check only, real fmp data are normalize first then imputation)

In [ ]:
sos run pipeline/phenotype_imputation.ipynb EBMF \
    --phenoFile output/leafcutter2/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.txt \
    --cwd output/withoutnorm_imput \
    --prior ebnm_point_laplace --varType 1 \
    --container oras://ghcr.io/cumc/factor_analysis_apptainer:latest \
    --mem 40G \
    --numThreads 20 \
    --walltime 100h \
    -c ~/env_files/csg.yml 

In [ ]:
sos run  ~/codes/xqtl-pipeline/pipeline/splicing_normalization.ipynb leafcutter_qqnorm \
    --cwd output/withoutnorm_imput/ \
    --qced-data output/withoutnorm_imput/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.imputed.bed.gz \
    --container oras://ghcr.io/cumc/leafcutter_apptainer:latest 

### Post-process of leafcutter outputs for them to be TensorQTL ready
*  `input`: output of the previous two steps and the gtf file.
*  `output`: a file in bed format end with "formated.bed.gz" 

#### map to genes and annotate

In [21]:
#must give realpath to sample_participant_lookup
# gene_annotation is the version on https://github.com/cumc/xqtl-protocol/blob/f9f0b0788b285e7692c0b398669e64a7c82883f7/code/data_preprocessing/phenotype/gene_annotation.ipynb
sos run  pipeline/gene_annotation.ipynb annotate_leafcutter_isoforms \
    --cwd output/normalize_impute \
    --intron_count output/leafcutter2/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz \
    --phenoFile output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.gz \
    --annotation_gtf /mnt/vast/hpc/csg/xqtl_workflow_testing/finalizing/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.gtf \
    --sample_participant_lookup  output/leafcutter2/sample_map.remove_duplicates.txt.rnaseq \
    --map_stra region \
    --container  oras://ghcr.io/cumc/bioinfo_apptainer:latest --mem 100G


In [ ]:
#above command always complaining not enough memory, which should not be the case
import pandas as pd
import numpy as np
import qtl.io
from pathlib import Path
# Load data
tss_df = qtl.io.gtf_to_tss_bed("/mnt/vast/hpc/csg/xqtl_workflow_testing/finalizing/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.gtf")
bed_df = pd.read_csv("~/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.gz", sep='\t', skiprows=0)
bed_df.columns.values[0] = "#chr" # Temporary
sample_participant_lookup = Path("/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/leafcutter2/sample_map.remove_duplicates.txt.rnaseq")
cluster2gene_dict = pd.read_csv("~/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz.leafcutter.clusters_to_genes.txt", sep='\t', index_col=0).to_dict()
cluster2gene_dict = cluster2gene_dict['genes']
print('    ** assigning introns to gene mapping(s)')
n = 0
gene_bed_df = []
group_s = {}
for _,r in bed_df.iterrows():
    s = r['ID'].split(':')
    cluster_id = s[0]+':'+s[3]
    if cluster_id in cluster2gene_dict:
        gene_ids = cluster2gene_dict[cluster_id].split(',')
        for g in gene_ids:
            gi = r['ID']+':'+g
            gene_bed_df.append(tss_df.loc[g, ['chr', 'start', 'end']].tolist() + [gi] + r.iloc[4:].tolist())
            group_s[gi] = g
    else:
        n += 1
        
        
if n > 0:
    print(f'    ** discarded {n} introns without a gene mapping')

print('  * writing BED files for QTL mapping')
gene_bed_df = pd.DataFrame(gene_bed_df, columns=bed_df.columns)
# sort by TSS
gene_bed_df = gene_bed_df.groupby('#chr', sort=False, group_keys=False).apply(lambda x: x.sort_values('start'))
#rename the samples if they named by file name (simply pick the first element with [.])
gene_bed_df.columns = list(gene_bed_df.columns[:4]) + [name.split('.')[0] if 'junc' in name else name for name in gene_bed_df.columns[4:]]
# change sample IDs to participant IDs
if sample_participant_lookup.is_file():
    sample_participant_lookup_s = pd.read_csv(sample_participant_lookup, sep="\t", index_col=0, dtype={0:str,1:str})
    #gene_bed_df.rename(columns=sample_participant_lookup_s.to_dict(), inplace=True)
    # Create a dictionary mapping from sample_id to participant_id
    column_mapping = dict(zip(sample_participant_lookup_s.index, sample_participant_lookup_s['participant_id']))
    # Get the column names to be replaced
    column_names = gene_bed_df.columns[4:]
    # Replace the column names using the mapping dictionary
    #new_column_names = [column_mapping.get(col, 'missing_data') for col in column_names]
    #it should overlap with genotype in downstream anyways
    new_column_names = [column_mapping.get(col, col) for col in column_names]
    gene_bed_df.rename(columns=dict(zip(column_names, new_column_names)), inplace=True)

gene_bed_df = gene_bed_df.drop_duplicates()
# qtl.io.write_bed(gene_bed_df, "~/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.formated.bed.gz")
uncompressed_path = "~/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.formated.bed"
gene_bed_df.to_csv(uncompressed_path, sep='\t', index=False)
bgzip_path = "/mnt/vast/hpc/homes/rf2872/software/htslib-1.9/bgzip"
compressed_path = uncompressed_path + ".gz"
bgzip_command = f"{bgzip_path} -c {uncompressed_path} > {compressed_path}"
#run below with bash 
tabix_path = "/mnt/vast/hpc/homes/rf2872/software/htslib-1.9/tabix"
tabix_command = f"{tabix_path}  -p bed  {compressed_path}"
#tabix -p bed ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.formated.bed.gz
subprocess.run(bgzip_command, shell=True, check=True)
subprocess.run(tabix_command, shell=True, check=True)

gene_bed_df[['start', 'end']] = gene_bed_df[['start', 'end']].astype(np.int32)
gene_bed_df[gene_bed_df.columns[4:]] = gene_bed_df[gene_bed_df.columns[4:]].astype(np.float32)
group_s_df =  pd.Series(group_s).sort_values().reset_index()
group_s_df.columns = ['ID', 'gene'] 
group_s_df.to_csv('~/Work/leaf_cutter2/ROSMAP_DLPFC_2024/output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.formated..phenotype_group.txt', sep='\t', index=False, header=True)

#### partition

In [ ]:
sos run pipeline/phenotype_formatting.ipynb phenotype_by_chrom \
    --cwd output/data_preprocessing/phenotype_data/phenotype_by_chrom \
    --phenoFile output/normalize_impute/ROSMAP_DLPFC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.formated.bed.gz \
    --chrom `for i in {1..22}; do echo chr$i; done` \
    --container /mnt/vast/hpc/csg/containers_xqtl/bioinfo.sif  --mem 40G -s force
